In [1]:
# importing packages
import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np
from astropy.coordinates import search_around_sky, SkyCoord
from astropy import units as u
from sklearn.model_selection import train_test_split
pd.set_option('display.max_columns', 999)

Here we match the Stripe 82 light curve data with the DR14 catalog which has the ground truth masses.  We use astropy SkyCoord function with a .5 arcsecond tolerance in matching.

In [ ]:
# matching LC and mass data
dr14_BH = pd.read_csv('/Users/baseb/Desktop/Uni files/DLiP/DLiP_project/data/dr14_BH.csv')
# dr14_LC = pd.read_csv('/Users/baseb/Desktop/Uni files/DLiP/DLiP_project/data/dr14_LC.csv')
dr14_LC = pd.read_csv('/Users/baseb/Desktop/Uni files/DLiP/DLiP_project/data/clean_stripe82.csv')

print(dr14_BH.shape)
print(dr14_LC.shape)

# quasar catalog redshift more reliable
dr14_LC = dr14_LC.drop(columns=['z'])

# # Add suffixes so duplicate column names stay distinct
# dr14_BH = dr14_BH.add_suffix('_BH')
# dr14_LC = dr14_LC.add_suffix('_LC')

# Match data attributes in the 2 data sets using astropy's SkyCoord
COORD1 = SkyCoord(dr14_BH['ra'], dr14_BH['dec'], frame='icrs', unit='deg')
COORD2 = SkyCoord(dr14_LC['ra'], dr14_LC['dec'], frame='icrs', unit='deg')

#in casse we have specified which columns come from which dataset.
# COORD1 = SkyCoord(dr14_BH['ra_BH'], dr14_BH['dec_BH'], frame='icrs', unit='deg')
# COORD2 = SkyCoord(dr14_LC['ra_LC'], dr14_LC['dec_LC'], frame='icrs', unit='deg')

IDX1, IDX2, OTHER1, OTHER2 = search_around_sky(COORD1, COORD2, seplimit=0.5 * u.arcsec)

# Generating columns for the matched
X_TRAIN = []
for i in range(len(IDX1)):
    # result = dr14_BH.iloc[IDX1[i]].append(dr14_LC.iloc[IDX2[i]]) # Original code, but append is deprecated
    result = pd.concat([dr14_BH.iloc[IDX1[i]], dr14_LC.iloc[IDX2[i]]])
    X_TRAIN.append(result)
X_TRAIN = pd.concat(X_TRAIN, axis=1)
X_TRAIN = X_TRAIN.T

X_TRAIN = X_TRAIN.loc[:, ~X_TRAIN.columns.str.contains('^Unnamed')]

(417618, 8)
(9258, 5)


In [16]:
display(X_TRAIN)

,SDSS_ID,ID,ra,dec,Mass,z,ERR,M_i,ID,ra,dec,BH_mass
0,4218-55479-0334,4218-55479-0334,3.071202,-0.910459,9.453132,3.602,0.096399,-27.210337,67243.0,3.071199,-0.910472,9.977
1,4239-55458-0110,4239-55458-0110,39.673284,-0.417003,9.093119,2.37,0.148789,-26.886169,2011765.0,39.673283,-0.417002,0.0
2,3651-55247-0177,3651-55247-0177,41.461633,-0.724516,9.553341,2.125,0.107519,-25.812664,2513385.0,41.461647,-0.724498,0.0
3,0687-52518-0383,0687-52518-0383,3.074126,0.554459,8.701729,1.3098,0.228644,-25.259472,89008.0,3.074112,0.554457,8.334
4,4226-55475-0777,4226-55475-0777,17.016002,0.113177,9.190273,2.761,0.046318,-27.111219,705845.0,17.015995,0.113198,-1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
7735,1486-52993-0456,1486-52993-0456,353.27142,0.706665,8.605764,1.011,0.14704,-23.670454,279962.0,353.271423,0.70666,0.0
7736,0684-52523-0052,0684-52523-0052,358.75743,-1.140242,8.724409,1.0863,0.060834,-25.567537,3936619.0,358.757416,-1.140248,8.709
7737,1963-54331-0558,1963-54331-0558,323.99881,-0.252224,8.984379,1.0524,0.088094,-24.37982,2227719.0,323.998802,-0.252199,-1.0
7738,1963-54331-0597,1963-54331-0597,324.04222,-0.07143,8.779519,0.7268,0.182696,-23.485292,2348443.0,324.042213,-0.071441,-1.0


We can compare the RA/DEC values on a few rows to confirm that the data matching happened correctly.

In [17]:
# remove repeat columns
# X_TRAIN = X_TRAIN.drop(columns = ['SDSS_ID', 'spec_mjd', 'ID'])
X_TRAIN = X_TRAIN.drop(columns = ['SDSS_ID'])
X_TRAIN.head()

,ID,ra,dec,Mass,z,ERR,M_i,ID,ra,dec,BH_mass
0,4218-55479-0334,3.071202,-0.910459,9.453132,3.602,0.096399,-27.210337,67243.0,3.071199,-0.910472,9.977
1,4239-55458-0110,39.673284,-0.417003,9.093119,2.37,0.148789,-26.886169,2011765.0,39.673283,-0.417002,0.0
2,3651-55247-0177,41.461633,-0.724516,9.553341,2.125,0.107519,-25.812664,2513385.0,41.461647,-0.724498,0.0
3,0687-52518-0383,3.074126,0.554459,8.701729,1.3098,0.228644,-25.259472,89008.0,3.074112,0.554457,8.334
4,4226-55475-0777,17.016002,0.113177,9.190273,2.761,0.046318,-27.111219,705845.0,17.015995,0.113198,-1.0


In [22]:
# convert to numeric
# X_TRAIN = X_TRAIN.apply(pd.to_numeric, errors='ignore')

def to_numeric(s):
    try:
        return pd.to_numeric(s, errors='raise')
    except ValueError:
        return s

X_TRAIN = X_TRAIN.apply(to_numeric)

After matching the data and cleaning black hole masses, we are left with 20549 objects total for the ML pipeline

In [23]:
print(X_TRAIN.shape)

(7740, 11)


We split the data into an 85% training and 15% testing set.

In [26]:
# split data
train, test = train_test_split(X_TRAIN, test_size=0.15)

# check
test.shape[0] + train.shape[0] == X_TRAIN.shape[0]

True

In [27]:
# X_TRAIN.to_csv('../../data/matched_dr14.csv')
X_TRAIN.to_csv('/Users/baseb/Desktop/Uni files/DLiP/DLiP_project/data/matched_dr14.csv')

In [28]:
train = train.dropna()
test = test.dropna()
# train.to_csv('../../data/TRAIN_dr14.csv')
train.to_csv('/Users/baseb/Desktop/Uni files/DLiP/DLiP_project/data/TRAIN_dr14.csv')
# test.to_csv('../../data/TEST_dr14.csv')
test.to_csv('/Users/baseb/Desktop/Uni files/DLiP/DLiP_project/data/TEST_dr14.csv')